In [1]:
import os
import sys
from dataclasses import dataclass

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.get_objective import get_objective

In [2]:
# === Configuration === (you edit here) ===
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "ridge"
    data_id: str = "042"
    n_folds: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 10
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"


cfg = Config()

opts = {
    "earlys_stopping_rounds": 5,
    "max_epochs": 20,
    "min_epochs": 4,
}

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# === Build & Run (frozen) ===
# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


create_objective = get_objective(cfg.model_name)
objective = create_objective(
    cfg.data_id,
    seed=cfg.seed,
    n_folds=cfg.n_folds,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    n_jobs=1,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner
)

[I 2025-09-30 21:25:46,475] Using an existing study with name 'ridge-042' instead of creating a new one.


  0%|          | 0/10 [00:00<?, ?it/s]

Fold Col: 5fold-s42
Free CPU Mem: 10.37 GB
Free GPU Mem: 6.95 GB


[W 2025-09-30 21:26:26,735] Trial 4 failed with parameters: {'alpha': 0.31489116479568624} because of the following error: AttributeError("'ndarray' object has no attribute 'to_numpy'").
Traceback (most recent call last):
  File "/home/hanse/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "/home/hanse/kaggle/binary-bank/src/models/ridge/ridge_objective.py", line 76, in objective
    score = trainer.fit_one_fold(
        fold_idx,
        loggers=[WandbLogger(run=run)]
    )
  File "/home/hanse/kaggle/binary-bank/src/models/ridge/ridge_cv_trainer.py", line 413, in fit_one_fold
    pred = model.predict(X_valid).to_numpy()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'ndarray' object has no attribute 'to_numpy'
[W 2025-09-30 21:26:26,751] Trial 4 failed with value None.


AttributeError: 'ndarray' object has no attribute 'to_numpy'